# Rural Road Extraction - MobileViT v2 Training Pipeline (Canopy-Resilient, Collapse-Guarded)

Self-contained notebook for Kaggle GPU environments (T4 / P100 / V100). No repo checkout required
-- every cell below is standalone, mirroring `backend/src/` and `backend/scripts/train.py` 1:1.

### What this notebook fixes vs. the previous run

A prior local run of this exact pipeline collapsed: the model converged to predicting **~90% of every
tile as "road"** (a uniform gray blob, not a road-shaped prediction at all), yet it was saved as the
"best" checkpoint. Two things caused that, and both are fixed here:

1. **`pos_weight` pushed the fresh decoder toward over-predicting positive.** The `AttentionGate` skip
   connections are new, randomly-initialized layers (this is a from-scratch retrain, not a fine-tune),
   and a `pos_weight=3.0` class-imbalance term made "call everything road" a cheap way to lower the BCE
   loss early in training. Default is now **`POS_WEIGHT = 2.0`**, and `--auto_pos_weight`-style
   estimation is clamped to `MAX_POS_WEIGHT` so it can never re-introduce this.
2. **The checkpoint metric could be gamed.** Checkpointing selected the epoch with the lowest raw
   `val_cldice` (soft centerline-Dice loss). We verified directly that a degenerate "80% probability
   everywhere" prediction scores `cldice_loss ≈ 0.11` -- only slightly worse than a *correct* sparse
   prediction's `≈ 0.01` -- because clDice's sensitivity term saturates near 1.0 once the prediction
   blankets the image. Meanwhile Dice/BCE correctly score that same degenerate prediction as
   near-worst. So a blob-collapsed epoch looked like the *best* epoch by the old criterion.
   **Fix:** checkpointing now selects on **validation IoU** (bounded, punished by both false positives
   and false negatives -- can't be gamed by covering the tile), and is **hard-gated**: any epoch whose
   predicted positive-pixel fraction exceeds `MAX_POS_FRAC` is rejected outright as a collapse,
   regardless of what any loss metric says.

### Canopy resilience (the actual goal: connect roads occluded by tree canopy)

- **`CanopyShadowDropout`** augmentation: soft, green-tinted, Gaussian-blurred occlusion patches
  simulate tree-canopy shadow during training without touching the ground-truth mask, so the model
  learns to keep predicting the road under partial occlusion.
- **Optional warm start (`INIT_FROM`)**: partially loads a prior good checkpoint (e.g. the older,
  non-collapsed `best_model_new.pth`) so the new `AttentionGate` layers train on top of an
  encoder/decoder that already knows where roads are, instead of a fully random init. This measurably
  prevents the collapse (validated locally: random init opened at ~90% positive-pixel fraction;
  warm-started training opened at ~2%, already road-shaped).
- **Post-training inference cell**: runs hysteresis thresholding + graph-based canopy-gap bridging on
  the freshly trained model and plots raw probability heatmap vs. final connected mask, so you can see
  directly, in this notebook, that predictions are sparse/road-shaped (not blobby) and that gaps under
  canopy get bridged into a single connected road.

### Architecture (unchanged -- this was never the problem)
1. **Backbone:** Ultra-lightweight MobileViT v2 encoder-decoder (~1.6M params at width_mult=1.0).
2. **Strip Convolutions (`StripConv`):** 1x3 + 3x1 directional filters, a road-shaped inductive prior.
3. **Channel Shift (`ChannelShift`):** zero-parameter receptive field expansion.
4. **Attention-Gated skip connections (`AttentionGate`):** suppresses non-road background clutter.
5. **Topology loss:** weighted BCE + SoftDice + SoftClDice, non-linear front-loaded alpha decay.

In [ ]:
!pip install gdown albumentations opencv-python-headless scipy scikit-image -q

import gdown
import os

# Dataset Download (DeepGlobe Rural Roads Archive)
file_id = '1OI0XJ1-ejxd0JS45hBzJbAYe9_2hJJwN'
url = f'https://drive.google.com/uc?id={file_id}'
output = '/kaggle/working/archive.zip'

if not os.path.exists(output):
    print("Downloading dataset...")
    gdown.download(url, output, quiet=False)
else:
    print("Dataset already downloaded.")

if not os.path.exists('/kaggle/working/dataset'):
    print("Extracting dataset...")
    !unzip -q /kaggle/working/archive.zip -d /kaggle/working/dataset
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

In [ ]:
import os
import sys
import math
import time
import json
import datetime
import random
from pathlib import Path
from typing import Dict, Tuple, List

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from scipy import ndimage
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

# --------------------------------------------------------------------------
# Paths Configuration
# --------------------------------------------------------------------------
TRAIN_IMG_DIR = '/kaggle/working/dataset/train'
TRAIN_MASK_DIR = '/kaggle/working/dataset/train'
VAL_IMG_DIR = '/kaggle/working/dataset/valid'
VAL_MASK_DIR = '/kaggle/working/dataset/valid'
OUTPUT_DIR = '/kaggle/working/model'

# Optional warm-start checkpoint. If you attach the older, known-good model as a
# Kaggle "Dataset" input, point this at it, e.g.
#   INIT_FROM = '/kaggle/input/road-extraction-checkpoints/best_model_new.pth'
# Leave as None to train fully from scratch (still safe -- collapse is now gated).
INIT_FROM = None

# --------------------------------------------------------------------------
# Training Configuration
# --------------------------------------------------------------------------
EPOCHS = 100
BATCH_SIZE = 16
LR = 1e-3
WIDTH_MULT = 1.0
NUM_WORKERS = min(4, os.cpu_count() or 2)
RUN_NAME = "kaggle-mobilevit-v2-fixed-run"
SEED = 42
GRAD_CLIP_NORM = 1.0       # 0 disables clipping

# --- Class imbalance ---
POS_WEIGHT = 2.0            # Lowered from 3.0 -- was aggressive enough to bias a
                             # freshly-initialized decoder toward over-predicting road.
AUTO_POS_WEIGHT = False     # If True, estimate from data, clamped to MAX_POS_WEIGHT below.
MAX_POS_WEIGHT = 3.0        # Upper clamp for AUTO_POS_WEIGHT (prevents runaway weighting).

# --- Loss shape / decay ---
ALPHA_START = 0.5           # Balanced start weight for BCE
ALPHA_END = 0.15            # Floor weight for BCE
DECAY_POWER = 0.5           # Non-linear decay power for fast clDice activation
DICE_WEIGHT = 0.35          # SoftDice loss weight -- directly penalizes false positives
ALPHA_DECAY_EPOCHS = 40     # Fixed schedule length for alpha decay regardless of total EPOCHS

# --- Checkpointing / collapse guard ---
PATIENCE = 20                # Early stopping patience on val_iou (was val_cldice -- gameable)
MAX_POS_FRAC = 0.20          # Hard gate: never checkpoint if predicted positive fraction exceeds this.
                              # Real road coverage in DeepGlobe-style tiles is typically a few percent;
                              # 20% is a generous ceiling that still catches blob collapse.

In [ ]:
def _wandb_available() -> bool:
    api_key = os.environ.get("WANDB_API_KEY", "").strip()
    if not api_key:
        return False
    try:
        import wandb
        return True
    except ImportError:
        return False

class WandbLogger:
    def __init__(
        self,
        project: str = "rural-road-extraction",
        run_name: str | None = None,
        config: dict | None = None,
        output_dir: str = "outputs",
        tags: list | None = None,
    ):
        self._use_wandb = _wandb_available()
        self._run_name = run_name or f"run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self._output_dir = Path(output_dir)
        self._output_dir.mkdir(parents=True, exist_ok=True)
        self._local_log_path = self._output_dir / "wandb_local.json"
        self._local_records: list[dict] = []
        self._step = 0

        if self._use_wandb:
            try:
                import wandb
                self._run = wandb.init(
                    project=project,
                    name=self._run_name,
                    config=config or {},
                    tags=tags or [],
                    reinit=True,
                )
                print(f"[WandbLogger] Connected to W&B project='{project}' run='{self._run_name}'")
            except Exception as e:
                print(f"[WandbLogger] W&B init failed ({e}), falling back to local logging.")
                self._use_wandb = False
                self._run = None
        else:
            self._run = None
            print(f"[WandbLogger] W&B not available. Logging locally to {self._local_log_path}")

    def log_metrics(self, metrics: dict, step: int | None = None) -> None:
        if step is None:
            self._step += 1
            step = self._step
        record = {"step": step, "timestamp": time.time(), **metrics}
        if self._use_wandb:
            try:
                import wandb
                wandb.log(metrics, step=step)
            except Exception:
                pass
        self._local_records.append(record)
        self._flush_local()

    def log_artifact(self, file_path: str, artifact_type: str = "model", name: str | None = None) -> None:
        file_path = str(file_path)
        name = name or Path(file_path).stem
        if self._use_wandb:
            try:
                import wandb
                artifact = wandb.Artifact(name=name, type=artifact_type)
                artifact.add_file(file_path)
                self._run.log_artifact(artifact)
            except Exception:
                pass
        else:
            self._local_records.append({"artifact": file_path, "type": artifact_type, "timestamp": time.time()})
            self._flush_local()

    def log_config(self, config: dict) -> None:
        if self._use_wandb and self._run is not None:
            try:
                import wandb
                wandb.config.update(config)
            except Exception:
                pass
        self._local_records.append({"config": config, "timestamp": time.time()})
        self._flush_local()

    def log_summary(self, summary: dict) -> None:
        if self._use_wandb and self._run is not None:
            try:
                import wandb
                for k, v in summary.items():
                    wandb.run.summary[k] = v
            except Exception:
                pass
        self._local_records.append({"summary": summary, "timestamp": time.time()})
        self._flush_local()

    def finish(self) -> None:
        if self._use_wandb and self._run is not None:
            try:
                import wandb
                wandb.finish()
            except Exception:
                pass
        self._flush_local()

    def _flush_local(self) -> None:
        try:
            with open(self._local_log_path, "w") as f:
                json.dump(self._local_records, f, indent=2, default=str)
        except Exception:
            pass

In [ ]:
class DeepGlobeDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.ids = []
        if os.path.exists(image_dir) and os.path.exists(mask_dir):
            for f in os.listdir(image_dir):
                if f.endswith(".jpg"):
                    img_id = f.split("_")[0]
                    mask_name = f"{img_id}_mask.png"
                    if os.path.exists(os.path.join(mask_dir, mask_name)):
                        self.ids.append(img_id)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        img_id = self.ids[index]
        img_path = os.path.join(self.image_dir, f"{img_id}_sat.jpg")
        mask_path = os.path.join(self.mask_dir, f"{img_id}_mask.png")

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((256, 256, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        else:
            mask = None

        if mask is None:
            h, w = image.shape[:2]
            mask = np.zeros((h, w), dtype=np.uint8)

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        if torch.is_tensor(mask):
            mask = mask.to(dtype=torch.float32) / 255.0
        else:
            mask = torch.tensor(mask, dtype=torch.float32) / 255.0

        mask = mask.unsqueeze(0)
        return image, mask

def estimate_pos_weight(dataset: Dataset, max_samples: int = 500, max_pos_weight: float = 3.0) -> float:
    """
    Measures class imbalance over dataset masks. Returns (negative/positive) pixel
    ratio for BCEWithLogitsLoss pos_weight, clamped to max_pos_weight -- the raw
    ratio for road masks (often 15-30x) is aggressive enough on its own to push a
    freshly-initialized decoder into a blanket-positive collapse.
    """
    pos, total = 0, 0
    num_samples = min(len(dataset), max_samples)
    print(f"Estimating pos_weight across {num_samples} dataset samples...")
    for i in range(num_samples):
        _, mask = dataset[i]
        pos += float(mask.sum().item())
        total += float(mask.numel())
    neg = total - pos
    raw_ratio = neg / max(pos, 1.0)
    ratio = min(raw_ratio, max_pos_weight)
    print(f"Estimated dataset pos_weight ratio: {raw_ratio:.2f} (clamped to {ratio:.2f}, cap={max_pos_weight})")
    return ratio

class CanopyShadowDropout(A.ImageOnlyTransform):
    """
    Simulates tree-canopy occlusion: darkened, desaturated, soft-edged green-tinted
    patches instead of unrealistic solid-black holes. Mask is left untouched, so the
    model is explicitly trained to keep predicting road under partial canopy cover.
    """
    def __init__(
        self,
        max_holes: int = 6,
        max_size: int = 48,
        min_size: int = 16,
        darken_range: tuple = (0.25, 0.55),
        always_apply: bool = False,
        p: float = 0.5,
    ):
        super().__init__(always_apply, p)
        self.max_holes = max_holes
        self.max_size = max_size
        self.min_size = min_size
        self.darken_range = darken_range

    def apply(self, img: np.ndarray, **params) -> np.ndarray:
        img = img.copy()
        h, w = img.shape[:2]
        n_holes = random.randint(1, self.max_holes)
        for _ in range(n_holes):
            hh = random.randint(self.min_size, self.max_size)
            ww = random.randint(self.min_size, self.max_size)
            y = random.randint(0, max(h - hh, 1))
            x = random.randint(0, max(w - ww, 1))
            patch = img[y:y + hh, x:x + ww].astype(np.float32)
            darken = random.uniform(*self.darken_range)
            green_tint = np.array([0.85, 1.0, 0.8], dtype=np.float32)
            patch = patch * darken * green_tint
            patch = cv2.GaussianBlur(patch, (5, 5), 0)
            img[y:y + hh, x:x + ww] = np.clip(patch, 0, 255).astype(img.dtype)
        return img

def get_train_transforms():
    return A.Compose([
        A.RandomCrop(width=256, height=256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.3),
        CanopyShadowDropout(max_holes=6, max_size=48, min_size=16, p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(height=256, width=256),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
        ToTensorV2(),
    ])

In [ ]:
# --------------------------------------------------------------------------
# Loss functions.
#
# NOTE on SoftClDiceLoss: its sensitivity term uses raw sigmoid(pred) against the
# target skeleton, so it saturates near 1.0 once a prediction blankets the image --
# it is NOT, by itself, a reliable signal that a prediction is good. It stays in the
# combined training loss (it genuinely helps connectivity once precision is already
# reasonable), but checkpoint SELECTION later in this notebook deliberately does NOT
# use it -- see the val_iou + MAX_POS_FRAC gate in the training loop.
# --------------------------------------------------------------------------

def soft_erode(img: torch.Tensor) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    p_v = -F.max_pool2d(-img, kernel_size=(3, 1), stride=1, padding=(1, 0))
    p_h = -F.max_pool2d(-img, kernel_size=(1, 3), stride=1, padding=(0, 1))
    return torch.min(p_v, p_h)

def soft_dilate(img: torch.Tensor) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    return F.max_pool2d(img, kernel_size=3, stride=1, padding=1)

def soft_open(img: torch.Tensor) -> torch.Tensor:
    return soft_dilate(soft_erode(img))

def soft_skel(img: torch.Tensor, num_iter: int = 10) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    img1 = soft_open(img)
    skel = F.relu(img - img1)
    for _ in range(num_iter):
        img = soft_erode(img)
        img1 = soft_open(img)
        delta = F.relu(img - img1)
        skel = skel + F.relu(delta - skel * delta)
    return skel

class SoftClDiceLoss(nn.Module):
    def __init__(self, num_iter: int = 10, smooth: float = 1.0):
        super().__init__()
        self.num_iter = num_iter
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)
        if target.dim() == 3:
            target = target.unsqueeze(1)

        skel_pred = soft_skel(torch.sigmoid(pred), self.num_iter)
        skel_target = soft_skel(target, self.num_iter)

        tprec_num = (skel_pred * target).sum(dim=(1, 2, 3)) + self.smooth
        tprec_den = skel_pred.sum(dim=(1, 2, 3)) + self.smooth
        tprec = tprec_num / tprec_den

        tsens_num = (skel_target * torch.sigmoid(pred)).sum(dim=(1, 2, 3)) + self.smooth
        tsens_den = skel_target.sum(dim=(1, 2, 3)) + self.smooth
        tsens = tsens_num / tsens_den

        cl_dice = 2.0 * (tprec * tsens) / (tprec + tsens + 1e-7)
        return (1.0 - cl_dice).mean()

class SoftDiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred = torch.sigmoid(pred)
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)
        if target.dim() == 3:
            target = target.unsqueeze(1)
        intersection = (pred * target).sum(dim=(1, 2, 3))
        cardinality = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return (1.0 - dice).mean()

class RoadExtractionLoss(nn.Module):
    def __init__(
        self,
        total_epochs: int,
        alpha_start: float = 0.5,
        alpha_end: float = 0.15,
        dice_weight: float = 0.35,
        num_iter: int = 10,
        smooth: float = 1.0,
        pos_weight: float = 2.0,
        decay_power: float = 0.5,
        alpha_decay_epochs: int | None = None,
    ):
        super().__init__()
        self.total_epochs = max(total_epochs, 1)
        self.decay_epochs = max(alpha_decay_epochs or total_epochs, 1)
        self.alpha_start = alpha_start
        self.alpha_end = alpha_end
        self.alpha = alpha_start
        self.dice_weight = dice_weight
        self.decay_power = decay_power

        pw_tensor = torch.tensor([pos_weight]) if isinstance(pos_weight, (int, float)) else pos_weight
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pw_tensor)
        self.dice = SoftDiceLoss(smooth=smooth)
        self.cldice = SoftClDiceLoss(num_iter=num_iter, smooth=smooth)

    def get_alpha(self) -> float:
        return self.alpha

    def update_alpha(self, epoch: int) -> float:
        frac = min(epoch / self.decay_epochs, 1.0) ** self.decay_power
        self.alpha = self.alpha_start - (self.alpha_start - self.alpha_end) * frac
        self.alpha = max(self.alpha_end, self.alpha)
        return self.alpha

    def forward(
        self,
        logits: torch.Tensor,
        target: torch.Tensor,
        return_components: bool = False,
    ) -> torch.Tensor | Tuple[torch.Tensor, Dict[str, float]]:
        if self.bce.pos_weight is not None and self.bce.pos_weight.device != logits.device:
            self.bce.pos_weight = self.bce.pos_weight.to(logits.device)

        with torch.amp.autocast(device_type=logits.device.type, enabled=False):
            logits_f32 = logits.float()
            target_f32 = target.float()
            bce_loss = self.bce(logits_f32, target_f32)
            dice_loss = self.dice(logits_f32, target_f32)
            cldice_loss = self.cldice(logits_f32, target_f32)

        cldice_w = max(0.0, 1.0 - self.alpha - self.dice_weight)
        total_loss = self.alpha * bce_loss + self.dice_weight * dice_loss + cldice_w * cldice_loss

        if return_components:
            components = {
                "total_loss": total_loss.item(),
                "bce_loss": bce_loss.item(),
                "dice_loss": dice_loss.item(),
                "cldice_loss": cldice_loss.item(),
                "alpha": self.alpha,
            }
            return total_loss, components
        return total_loss

def avg_component_count(pred_masks: torch.Tensor, threshold: float = 0.5) -> float:
    counts = []
    preds_np = (torch.sigmoid(pred_masks) > threshold).cpu().numpy().astype("uint8")
    for m in preds_np:
        if m.ndim == 3:
            m = m[0]
        _, n = ndimage.label(m)
        counts.append(n)
    return float(sum(counts) / max(len(counts), 1))

In [ ]:
def _make_divisible(v: float, divisor: int = 8, min_value: int = None) -> int:
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

class StripConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv_h = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 3), stride=(1, stride), padding=(0, 1), bias=False)
        self.bn_h = nn.BatchNorm2d(out_channels)
        self.conv_v = nn.Conv2d(out_channels, out_channels, kernel_size=(3, 1), stride=(stride, 1), padding=(1, 0), bias=False)
        self.bn_v = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.bn_h(self.conv_h(x)))
        x = self.act(self.bn_v(self.conv_v(x)))
        return x

class ChannelShift(nn.Module):
    def __init__(self, shift_pixels: int = 2, shift_fraction: float = 0.25):
        super().__init__()
        self.shift_pixels = shift_pixels
        self.shift_fraction = shift_fraction

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        s = self.shift_pixels
        n_shifted = int(C * self.shift_fraction)
        per_dir = n_shifted // 4
        if per_dir == 0 or s == 0:
            return x
        c_up    = F.pad(x[:, 0 * per_dir : 1 * per_dir][:, :, s:, :],     (0, 0, 0, s))
        c_down  = F.pad(x[:, 1 * per_dir : 2 * per_dir][:, :, :-s, :],  (0, 0, s, 0))
        c_left  = F.pad(x[:, 2 * per_dir : 3 * per_dir][:, :, :, s:],   (0, s, 0, 0))
        c_right = F.pad(x[:, 3 * per_dir : 4 * per_dir][:, :, :, :-s], (s, 0, 0, 0))
        c_id    = x[:, 4 * per_dir :]
        return torch.cat([c_up, c_down, c_left, c_right, c_id], dim=1)

class InvertedResidual(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, expand_ratio: int = 2):
        super().__init__()
        mid = in_channels * expand_ratio
        self.use_residual = (stride == 1 and in_channels == out_channels)
        layers = []
        if expand_ratio != 1:
            layers.extend([nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU(inplace=True)])
        layers.extend([
            nn.Conv2d(mid, mid, 3, stride=stride, padding=1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True),
            nn.Conv2d(mid, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.conv(x) if self.use_residual else self.conv(x)

class LinearSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dropout: float = 0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.qkv = nn.Linear(embed_dim, 2 * embed_dim + 1, bias=True)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.attn_drop = nn.Dropout(attn_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        qkv = self.qkv(x)
        q, k, v = qkv.split([self.embed_dim, 1, self.embed_dim], dim=-1)
        context_scores = self.attn_drop(F.softmax(k, dim=1))
        context_vector = (context_scores * v).sum(dim=1, keepdim=True)
        return self.out_proj(F.relu(q) * context_vector)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, ffn_ratio: float = 2.0, dropout: float = 0.0, attn_dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LinearSelfAttention(embed_dim, attn_dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        ffn_dim = int(embed_dim * ffn_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim), nn.SiLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim), nn.Dropout(dropout)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class MobileViTv2Block(nn.Module):
    def __init__(self, in_channels: int, transformer_dim: int, n_transformer_layers: int = 2, patch_size: int = 2, ffn_ratio: float = 2.0, dropout: float = 0.0, attn_dropout: float = 0.0, shift_pixels: int = 2, shift_fraction: float = 0.25):
        super().__init__()
        self.patch_h, self.patch_w = patch_size, patch_size
        self.channel_shift = ChannelShift(shift_pixels, shift_fraction)
        self.local_rep = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels), nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, transformer_dim, 1, bias=False), nn.BatchNorm2d(transformer_dim)
        )
        self.transformers = nn.Sequential(*[TransformerBlock(transformer_dim, ffn_ratio, dropout, attn_dropout) for _ in range(n_transformer_layers)])
        self.post_norm = nn.LayerNorm(transformer_dim)
        self.proj = nn.Sequential(nn.Conv2d(transformer_dim, in_channels, 1, bias=False), nn.BatchNorm2d(in_channels))
        self.fusion = nn.Sequential(nn.Conv2d(2 * in_channels, in_channels, 1, bias=False), nn.BatchNorm2d(in_channels), nn.SiLU(inplace=True))

    def _unfold(self, x: torch.Tensor):
        B, C, H, W = x.shape
        ph, pw = self.patch_h, self.patch_w
        n_h, n_w = H // ph, W // pw
        x = x.reshape(B, C, n_h, ph, n_w, pw).permute(0, 3, 5, 1, 2, 4).reshape(B * ph * pw, C, n_h * n_w).permute(0, 2, 1)
        return x, (B, n_h, n_w)

    def _fold(self, x: torch.Tensor, info: tuple, C: int):
        B, n_h, n_w = info
        ph, pw = self.patch_h, self.patch_w
        return x.permute(0, 2, 1).reshape(B, ph, pw, C, n_h, n_w).permute(0, 3, 4, 1, 5, 2).reshape(B, C, n_h * ph, n_w * pw)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        ph, pw = self.patch_h, self.patch_w
        pad_h = (ph - H % ph) % ph
        pad_w = (pw - W % pw) % pw
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        identity = x
        local_out = self.local_rep(self.channel_shift(x))
        tokens, fold_info = self._unfold(local_out)
        tokens = self.post_norm(self.transformers(tokens))
        global_out = self.proj(self._fold(tokens, fold_info, local_out.shape[1]))
        fused = self.fusion(torch.cat([identity, global_out], dim=1))
        return fused[:, :, :H, :W] if (pad_h > 0 or pad_w > 0) else fused

class AttentionGate(nn.Module):
    def __init__(self, gate_channels: int, skip_channels: int, inter_channels: int = None):
        super().__init__()
        inter_channels = inter_channels or max(skip_channels // 2, 8)
        self.W_gate = nn.Sequential(nn.Conv2d(gate_channels, inter_channels, 1, bias=True), nn.BatchNorm2d(inter_channels))
        self.W_skip = nn.Sequential(nn.Conv2d(skip_channels, inter_channels, 1, bias=True), nn.BatchNorm2d(inter_channels))
        self.psi = nn.Sequential(nn.Conv2d(inter_channels, 1, 1, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.act = nn.SiLU(inplace=True)

    def forward(self, gate: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        g = self.W_gate(gate)
        s = self.W_skip(skip)
        attn = self.psi(self.act(g + s))
        return skip * attn

class MobileViT_v2(nn.Module):
    def __init__(self, num_classes: int = 1, width_mult: float = 1.0):
        super().__init__()
        def _c(channels: int) -> int:
            return _make_divisible(channels * width_mult)

        self.stem = StripConv(3, _c(32), stride=2)
        self.enc1 = InvertedResidual(_c(32), _c(64), stride=2)
        self.enc2_mvit = MobileViTv2Block(_c(64), transformer_dim=_c(96), n_transformer_layers=2)
        self.enc2_down = InvertedResidual(_c(64), _c(96), stride=2)
        self.enc3_mvit = MobileViTv2Block(_c(96), transformer_dim=_c(144), n_transformer_layers=2)
        self.enc3_down = InvertedResidual(_c(96), _c(128), stride=2)
        self.bottleneck = MobileViTv2Block(_c(128), transformer_dim=_c(192), n_transformer_layers=3)

        self.gate3 = AttentionGate(_c(128), _c(96))
        self.gate2 = AttentionGate(_c(96), _c(64))
        self.gate1 = AttentionGate(_c(64), _c(32))

        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec3 = StripConv(_c(128) + _c(96), _c(96))
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec2 = StripConv(_c(96) + _c(64), _c(64))
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec1 = StripConv(_c(64) + _c(32), _c(32))
        self.up0 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.head = nn.Conv2d(_c(32), num_classes, kernel_size=1, bias=True)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s1 = self.stem(x)
        e1 = self.enc1(s1)
        s2 = self.enc2_mvit(e1)
        e2 = self.enc2_down(s2)
        s3 = self.enc3_mvit(e2)
        e3 = self.enc3_down(s3)
        bn = self.bottleneck(e3)

        up3 = self.up3(bn)
        s3_gated = self.gate3(up3, s3)
        d3 = self.dec3(torch.cat([up3, s3_gated], dim=1))

        up2 = self.up2(d3)
        s2_gated = self.gate2(up2, s2)
        d2 = self.dec2(torch.cat([up2, s2_gated], dim=1))

        up1 = self.up1(d2)
        s1_gated = self.gate1(up1, s1)
        d1 = self.dec1(torch.cat([up1, s1_gated], dim=1))

        out = self.head(self.up0(d1))
        return out

    @property
    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_warm_start(model: nn.Module, checkpoint_path: str, device: torch.device) -> None:
    """
    Partially loads weights from a prior checkpoint (e.g. an older model without
    AttentionGate layers). Missing/unexpected keys are reported but not fatal -- this
    lets a new architecture start from an encoder/decoder that already knows where
    roads are, instead of a fully random init that pos_weight can push into a
    blanket-positive collapse before the topology losses can correct it.
    """
    print(f"--- Warm-starting from {checkpoint_path} ---")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['model_state_dict'] if (isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint) else checkpoint
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"    Loaded {len(state_dict) - len(unexpected)} matching tensors.")
    if missing:
        print(f"    Missing (randomly initialized): {missing}")
    if unexpected:
        print(f"    Unexpected (ignored): {unexpected}")


def train_one_epoch(epoch, model, dataloader, optimizer, scaler, loss_fn, logger, device, grad_clip_norm):
    model.train()
    total_loss = 0.0
    total_bce = 0.0
    total_cldice = 0.0
    current_alpha = loss_fn.update_alpha(epoch)

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss, components = loss_fn(logits, masks, return_components=True)

        scaler.scale(loss).backward()
        if grad_clip_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_bce += components['bce_loss']
        total_cldice += components['cldice_loss']
        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Alpha": f"{current_alpha:.2f}"})

        logger.log_metrics({
            "train/step_loss": loss.item(),
            "train/step_bce": components['bce_loss'],
            "train/step_cldice": components['cldice_loss'],
            "lr": optimizer.param_groups[0]['lr']
        })

    num_batches = len(dataloader)
    return total_loss / num_batches, total_bce / num_batches, total_cldice / num_batches

@torch.no_grad()
def validate(epoch, model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_bce = 0.0
    total_cldice = 0.0
    total_comp_count = 0.0
    total_iou = 0.0
    total_prec = 0.0
    total_pos_frac = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)

        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            preds = model(images)
            loss, components = loss_fn(preds, masks, return_components=True)

        total_loss += loss.item()
        total_bce += components['bce_loss']
        total_cldice += components['cldice_loss']
        total_comp_count += avg_component_count(preds)

        probs = torch.sigmoid(preds)
        pred_bin = (probs > 0.5).float()
        target_bin = (masks > 0.5).float()
        total_pos_frac += pred_bin.mean().item()
        inter = (pred_bin * target_bin).sum(dim=(1, 2, 3))
        union = (pred_bin + target_bin).clamp(0, 1).sum(dim=(1, 2, 3))
        total_iou += ((inter + 1e-7) / (union + 1e-7)).mean().item()
        total_prec += ((inter + 1e-7) / (pred_bin.sum(dim=(1, 2, 3)) + 1e-7)).mean().item()
        pbar.set_postfix({"Val Loss": f"{loss.item():.4f}"})

    num_batches = len(dataloader)
    avg_pos_frac = total_pos_frac / num_batches
    if avg_pos_frac > 0.10:
        print(f"⚠️ CANARY WARNING: High predicted positive pixel fraction ({avg_pos_frac*100:.1f}% > 10%). Check model precision!")
    return total_loss / num_batches, total_bce / num_batches, total_cldice / num_batches, total_comp_count / num_batches, total_iou / num_batches, total_prec / num_batches, avg_pos_frac

## Run Training

Checkpointing below selects on **validation IoU** and hard-rejects any epoch where the
predicted positive-pixel fraction exceeds `MAX_POS_FRAC` -- printed as `REJECTED` so you
can watch it happen live if a run starts to collapse. `val_cldice` is still logged for
visibility but never drives the save decision.

In [ ]:
def main():
    seed_everything(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("--- Validating Paths ---")
    for name, path in [
        ('Train Images', TRAIN_IMG_DIR),
        ('Train Masks', TRAIN_MASK_DIR),
        ('Val Images', VAL_IMG_DIR),
        ('Val Masks', VAL_MASK_DIR)
    ]:
        if not os.path.exists(path):
            print(f"⚠️ Warning: Path not found for {name}: {path}")
        else:
            print(f"✅ {name} path exists: {path}")

    logger = WandbLogger(
        project="rural-road-extraction",
        run_name=RUN_NAME,
        config={
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "width_mult": WIDTH_MULT,
            "num_workers": NUM_WORKERS,
            "pos_weight": POS_WEIGHT,
            "max_pos_frac": MAX_POS_FRAC,
            "init_from": INIT_FROM,
        },
        output_dir=OUTPUT_DIR
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dataset = DeepGlobeDataset(
        image_dir=TRAIN_IMG_DIR,
        mask_dir=TRAIN_MASK_DIR,
        transform=get_train_transforms()
    )
    val_dataset = DeepGlobeDataset(
        image_dir=VAL_IMG_DIR,
        mask_dir=VAL_MASK_DIR,
        transform=get_val_transforms()
    )

    if len(train_dataset) == 0:
        raise ValueError(f"No valid image/mask pairs found in training directory: {TRAIN_IMG_DIR}")

    pos_weight = POS_WEIGHT
    if AUTO_POS_WEIGHT:
        pos_weight = estimate_pos_weight(train_dataset, max_pos_weight=MAX_POS_WEIGHT)

    if len(val_dataset) == 0:
        print("⚠️ Warning: Validation dataset has no masks. Dynamically splitting training dataset (90/10) for validation...")
        rng = random.Random(SEED)
        all_ids = list(train_dataset.ids)
        rng.shuffle(all_ids)

        val_size = max(1, int(len(all_ids) * 0.1))
        train_ids = all_ids[val_size:]
        val_ids = all_ids[:val_size]

        train_dataset.ids = train_ids
        val_dataset.ids = val_ids
        val_dataset.image_dir = TRAIN_IMG_DIR
        val_dataset.mask_dir = TRAIN_MASK_DIR
        print(f"✅ Split dataset into {len(train_dataset)} training and {len(val_dataset)} validation samples.")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = MobileViT_v2(num_classes=1, width_mult=WIDTH_MULT).to(device)
    if INIT_FROM:
        load_warm_start(model, INIT_FROM, device)
    logger.log_config({"num_parameters": model.num_parameters})
    print(f"Model parameters: {model.num_parameters:,}")

    loss_fn = RoadExtractionLoss(
        total_epochs=EPOCHS,
        alpha_start=ALPHA_START,
        alpha_end=ALPHA_END,
        dice_weight=DICE_WEIGHT,
        pos_weight=pos_weight,
        decay_power=DECAY_POWER,
        alpha_decay_epochs=ALPHA_DECAY_EPOCHS
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    scaler = GradScaler(enabled=(device.type == 'cuda'))

    # See the collapse-guard note above the Loss cell: checkpointing is driven by
    # val_iou (bounded, directly punished by false positives/negatives) and hard-gated
    # by MAX_POS_FRAC, NOT by raw val_cldice.
    best_val_iou = -1.0
    patience_counter = 0

    for epoch in range(EPOCHS):
        train_loss, train_bce, train_cldice = train_one_epoch(
            epoch, model, train_loader, optimizer, scaler, loss_fn, logger, device, GRAD_CLIP_NORM
        )
        val_loss, val_bce, val_cldice, val_comp_count, val_iou, val_prec, val_pos_frac = validate(epoch, model, val_loader, loss_fn, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        logger.log_metrics({
            "epoch": epoch,
            "train/epoch_loss": train_loss,
            "train/epoch_bce": train_bce,
            "train/epoch_cldice": train_cldice,
            "val/epoch_loss": val_loss,
            "val/epoch_bce": val_bce,
            "val/epoch_cldice": val_cldice,  # logged for visibility only -- not used for checkpointing
            "val/component_count": val_comp_count,
            "val/iou": val_iou,
            "val/precision": val_prec,
            "val/positive_frac": val_pos_frac,
            "alpha": loss_fn.get_alpha(),
            "lr": current_lr
        }, step=epoch)

        print(f"Epoch [{epoch}/{EPOCHS-1}] - "
              f"Train Loss: {train_loss:.4f} (clDice: {train_cldice:.4f}) | "
              f"Val Loss: {val_loss:.4f} (clDice: {val_cldice:.4f}, IoU: {val_iou:.4f}, Prec: {val_prec:.4f}, PosFrac: {val_pos_frac*100:.1f}%) | "
              f"LR: {current_lr:.6f}")

        collapsed = val_pos_frac > MAX_POS_FRAC
        if collapsed:
            print(f"🚫 REJECTED checkpoint at epoch {epoch}: predicted positive fraction "
                  f"({val_pos_frac*100:.1f}%) exceeds MAX_POS_FRAC ({MAX_POS_FRAC*100:.1f}%). "
                  f"This looks like blob collapse, not a real road prediction -- not saving.")

        if (not collapsed) and val_iou > best_val_iou:
            best_val_iou = val_iou
            patience_counter = 0
            save_path = os.path.join(OUTPUT_DIR, "best_model_v2.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_cldice': val_cldice,
                'val_iou': best_val_iou,
                'val_precision': val_prec,
                'val_positive_frac': val_pos_frac,
            }, save_path)
            print(f"--> Saved new best model to {save_path} (Val IoU: {best_val_iou:.4f}, clDice: {val_cldice:.4f}, PosFrac: {val_pos_frac*100:.1f}%)")
            logger.log_artifact(save_path, artifact_type="model", name="best_model_v2")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"🛑 Early stopping triggered after {patience_counter} epochs without val_iou improvement at epoch {epoch}.")
                break

    logger.log_summary({"best_val_iou": best_val_iou})
    logger.finish()
    return os.path.join(OUTPUT_DIR, "best_model_v2.pth")

BEST_MODEL_PATH = main()

## Inference & Canopy Gap-Connection Postprocessing

This runs the just-trained model on held-out validation tiles and shows, side by side:

1. **Raw probability heatmap** -- proves the network itself outputs a sparse, road-shaped
   signal (not a uniform blob) after the fixes above.
2. **Hysteresis threshold** -- keeps any weak probability (down to `low_thresh`) that is
   connected to a high-confidence road pixel, recovering faint signal under canopy shadow
   without letting noise elsewhere in the tile turn on.
3. **Graph-based canopy-gap bridging** (`connect_canopy_gaps`) -- skeletonizes the mask,
   finds dead-end endpoints, and draws short connecting strokes between endpoints that face
   each other within `max_gap_dist` and are roughly collinear -- i.e. it reconnects a road
   that the model lost track of for a few dozen pixels under a tree, without arbitrarily
   fusing unrelated roads together (the angle/distance gate prevents that).

In [ ]:
try:
    from skimage.morphology import skeletonize
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False


def hysteresis_threshold(probs: np.ndarray, high_thresh: float = 0.35, low_thresh: float = 0.12) -> np.ndarray:
    """Keeps weak road probabilities (>= low_thresh) only if connected to a high-confidence
    (>= high_thresh) region -- recovers faint signal under canopy without admitting noise."""
    strong = probs >= high_thresh
    weak = (probs >= low_thresh) & (probs < high_thresh)

    total_candidate = (strong | weak).astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(total_candidate, connectivity=8)

    if num_labels <= 1:
        return (strong * 255).astype(np.uint8)

    strong_labels = np.unique(labels[strong])
    strong_labels = strong_labels[strong_labels > 0]

    keep_mask = np.isin(labels, strong_labels)
    output_mask = np.zeros_like(total_candidate, dtype=np.uint8)
    output_mask[keep_mask] = 255
    return output_mask


def find_skeleton_endpoints(skel: np.ndarray):
    """Finds endpoints in a 1-pixel binary skeleton and their outward tangent directions."""
    skel_bool = (skel > 0).astype(np.uint8)
    kernel = np.array([[1, 1, 1], [1, 10, 1], [1, 1, 1]], dtype=np.uint8)
    filtered = cv2.filter2D(skel_bool, -1, kernel)
    ey, ex = np.where(filtered == 11)

    endpoints = []
    h, w = skel.shape
    for y, x in zip(ey, ex):
        patch_y1, patch_y2 = max(0, y - 4), min(h, y + 5)
        patch_x1, patch_x2 = max(0, x - 4), min(w, x + 5)
        py, px = np.where(skel_bool[patch_y1:patch_y2, patch_x1:patch_x2] > 0)
        py = py + patch_y1
        px = px + patch_x1
        if len(py) > 1:
            dy = float(y - np.mean(py[py != y])) if any(py != y) else 0.0
            dx = float(x - np.mean(px[px != x])) if any(px != x) else 0.0
            norm = np.hypot(dx, dy)
            dir_vec = np.array([dx / norm, dy / norm]) if norm > 1e-5 else np.array([0.0, 0.0])
        else:
            dir_vec = np.array([0.0, 0.0])
        endpoints.append((y, x, dir_vec))
    return endpoints


def connect_canopy_gaps(binary_mask: np.ndarray, max_gap_dist: float = 220.0, max_angle_deg: float = 65.0, road_width: int = 6) -> np.ndarray:
    """Skeletonizes, then bridges facing dead-end endpoints across canopy gaps and connects
    dead-ends to nearby road segments (T-junction completion), gated by distance and angle
    so unrelated road fragments don't get fused together."""
    mask_out = binary_mask.copy()

    if HAS_SKIMAGE:
        skel = skeletonize(mask_out > 0).astype(np.uint8)
    elif hasattr(cv2, 'ximgproc'):
        skel = cv2.ximgproc.thinning(mask_out)
    else:
        skel = (mask_out > 0).astype(np.uint8)
        element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
        skel_acc = np.zeros(mask_out.shape, dtype=np.uint8)
        img_temp = skel.copy()
        while True:
            eroded = cv2.erode(img_temp, element)
            temp = cv2.dilate(eroded, element)
            temp = cv2.subtract(img_temp, temp)
            skel_acc = cv2.bitwise_or(skel_acc, temp)
            img_temp = eroded.copy()
            if cv2.countNonZero(img_temp) == 0:
                break
        skel = skel_acc

    endpoints = find_skeleton_endpoints(skel)
    n_pts = len(endpoints)
    if n_pts == 0:
        return mask_out

    connected_pairs = []
    connected_endpoints = set()
    cos_threshold = np.cos(np.radians(max_angle_deg))

    # Strategy A: endpoint-to-endpoint collinear & facing pair connection
    for i in range(n_pts):
        y1, x1, v1 = endpoints[i]
        best_j, best_score = None, float('inf')
        for j in range(i + 1, n_pts):
            y2, x2, v2 = endpoints[j]
            dist = float(np.hypot(x2 - x1, y2 - y1))
            if dist > max_gap_dist or dist < 5.0:
                continue
            gap_vec = np.array([(x2 - x1) / dist, (y2 - y1) / dist])
            dot1 = float(np.dot(v1, gap_vec)) if np.linalg.norm(v1) > 0 else 0.8
            dot2 = float(np.dot(v2, -gap_vec)) if np.linalg.norm(v2) > 0 else 0.8
            if dot1 >= cos_threshold and dot2 >= cos_threshold:
                alignment_penalty = (2.0 - dot1 - dot2) * 50.0
                score = dist + alignment_penalty
                if score < best_score:
                    best_score, best_j = score, j
        if best_j is not None:
            y2, x2, _ = endpoints[best_j]
            connected_pairs.append(((x1, y1), (x2, y2)))
            connected_endpoints.add(i)
            connected_endpoints.add(best_j)

    # Strategy B: endpoint-to-road-edge connection (T-junction completion)
    skel_y, skel_x = np.where(skel > 0)
    if len(skel_x) > 0:
        skel_pts = np.column_stack((skel_x, skel_y))
        for i in range(n_pts):
            if i in connected_endpoints:
                continue
            y1, x1, v1 = endpoints[i]
            if np.linalg.norm(v1) == 0:
                continue
            dists_to_skel = np.hypot(skel_pts[:, 0] - x1, skel_pts[:, 1] - y1)
            valid_idx = dists_to_skel > 25.0
            if not np.any(valid_idx):
                continue
            cand_pts, cand_dists = skel_pts[valid_idx], dists_to_skel[valid_idx]
            within_dist = cand_dists <= max_gap_dist
            if not np.any(within_dist):
                continue
            cand_pts, cand_dists = cand_pts[within_dist], cand_dists[within_dist]
            best_cand, best_cand_score = None, float('inf')
            for (cx, cy), d in zip(cand_pts, cand_dists):
                g_vec = np.array([(cx - x1) / d, (cy - y1) / d])
                dot = float(np.dot(v1, g_vec))
                if dot >= cos_threshold:
                    score = d + (1.0 - dot) * 60.0
                    if score < best_cand_score:
                        best_cand_score, best_cand = score, (cx, cy)
            if best_cand is not None:
                connected_pairs.append(((x1, y1), best_cand))

    for pt1, pt2 in connected_pairs:
        cv2.line(mask_out, pt1, pt2, 255, thickness=road_width)
    return mask_out

In [ ]:
import matplotlib.pyplot as plt

def run_inference_demo(model, image_paths, device, n_show=4):
    model.eval()
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    image_paths = image_paths[:n_show]
    fig, axes = plt.subplots(len(image_paths), 4, figsize=(18, 4.5 * len(image_paths)))
    if len(image_paths) == 1:
        axes = axes[None, :]

    for row, img_path in enumerate(image_paths):
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image_rgb.shape[:2]
        new_h, new_w = (h // 32) * 32, (w // 32) * 32
        resized = cv2.resize(image_rgb, (new_w, new_h))
        normed = (resized.astype(np.float32) / 255.0 - mean) / std
        input_tensor = torch.from_numpy(normed.transpose(2, 0, 1)).unsqueeze(0).float().to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = torch.sigmoid(logits).squeeze().cpu().numpy()

        pos_frac_raw = float((probs > 0.5).mean())
        mask_hyst = hysteresis_threshold(probs, high_thresh=0.35, low_thresh=0.12)
        kernel_close = np.ones((5, 5), np.uint8)
        mask_closed = cv2.morphologyEx(mask_hyst, cv2.MORPH_CLOSE, kernel_close, iterations=1)
        mask_connected = connect_canopy_gaps(mask_closed, max_gap_dist=220.0, max_angle_deg=65.0, road_width=6)

        status = "OK (sparse)" if pos_frac_raw < 0.20 else "COLLAPSE WARNING (blob-like)"
        axes[row, 0].imshow(resized); axes[row, 0].set_title(os.path.basename(img_path)); axes[row, 0].axis("off")
        axes[row, 1].imshow(probs, cmap="jet", vmin=0, vmax=1); axes[row, 1].set_title(f"Raw prob (>0.5: {pos_frac_raw*100:.1f}%, {status})"); axes[row, 1].axis("off")
        axes[row, 2].imshow(mask_hyst, cmap="gray"); axes[row, 2].set_title("Hysteresis threshold"); axes[row, 2].axis("off")
        axes[row, 3].imshow(mask_connected, cmap="gray"); axes[row, 3].set_title("+ canopy-gap bridging"); axes[row, 3].axis("off")

    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "inference_demo.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved demo figure to {out_path}")


# Load the checkpoint that training just selected and run the demo on a handful of
# validation tiles.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
demo_model = MobileViT_v2(num_classes=1, width_mult=WIDTH_MULT).to(device)
ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
demo_model.load_state_dict(ckpt['model_state_dict'])
demo_model.eval()
print(f"Loaded {BEST_MODEL_PATH} (epoch {ckpt.get('epoch')}, val_iou={ckpt.get('val_iou'):.4f}, "
      f"val_positive_frac={ckpt.get('val_positive_frac', float('nan'))*100:.1f}%)")

demo_dir = VAL_IMG_DIR if os.path.exists(VAL_IMG_DIR) else TRAIN_IMG_DIR
demo_images = sorted(
    os.path.join(demo_dir, f) for f in os.listdir(demo_dir) if f.endswith("_sat.jpg")
)
run_inference_demo(demo_model, demo_images, device, n_show=min(4, len(demo_images)))

## Summary

- Best checkpoint: `/kaggle/working/model/best_model_v2.pth`, selected on validation IoU and
  hard-gated against blob collapse (see the `REJECTED` lines in the training log above, if any
  fired).
- `wandb_local.json` in `OUTPUT_DIR` has the full per-epoch metric history (works with or
  without a W&B API key).
- If every epoch gets `REJECTED` and training never saves anything: lower `POS_WEIGHT` further
  (try `1.0`), or set `INIT_FROM` to a prior working checkpoint to warm-start instead of
  training the new `AttentionGate` layers from scratch.
- If IoU plateaus low but nothing collapses: the inference demo above is the fastest way to
  see whether the failure is "roads too broken under canopy" (tune `connect_canopy_gaps`'s
  `max_gap_dist` / `max_angle_deg`) vs. "roads missed entirely" (train longer / increase
  `CanopyShadowDropout` probability so the model sees more occlusion during training).